# Exploring SOMISANA Operational Ocean Forecasts

**A tutorial in operational oceanography**

If you use Colab instead of you local computer do File → Save a copy in Drive,right after opening the Colab link to make sure changes are saved.

This notebook is an introduction to the ocean forecast products produced
by **SOMISANA** — the *Sustainable Ocean Modelling Initiative: a South African Approach* (https://somisana.ac.za/). SOMISANA runs the [CROCO](https://www.croco-ocean.org/) regional ocean model operationally around South Africa and publishes the output through a THREDDS/ERDDAP catalog: https://catalog.somisana.ac.za/catalog/

This forecast is porvided hourly, but to avoid too high data storage you will download  data that has been temporally averaged to **daily forecast files** for **two regions** and **four model configurations**:

| | GFS atmospheric forcing | SAWS atmospheric forcing |
|---|---|---|
| **HYCOM** boundary/initial conditions | `HYCOM-GFS` | `HYCOM-SAWS` |
| **MERCATOR** boundary/initial conditions | `MERCATOR-GFS` | `MERCATOR-SAWS` |

for the regions:

- **SA-West** — the Benguela / Cape Town shelf region
- **SA-East** — a smaller domain around Cape Point / False Bay

CROCO (the regional model) is **nested** inside one of two global ocean
models (**HYCOM** or **Mercator**, which supply the open-boundary and initial conditions), and is forced at the surface by one of two atmospheric products (**GFS**, the NOAA/NCEP global forecast, or **SAWS**, the South African Weather Service regional forecast). Tides are added from **TPXO10**. You will compare the four combinations to explore *forecast uncertainty that comes from the choice of forcing data*.

The forecast from 3. September 2026 has been downloaded and placed here: 
[SOMISANA 3. September](https://ns9481k.web.sigma2.no/SOMISANA_forecasts)
This provides a 5-day hindcast ahead of 3. September and a 3 and 5-day forecast (for SAWS and GFS resoectively).

- `croco_avg_t3.nc` — the original **hourly** model output, regridded onto a
  regular lon/lat/depth grid (CROCO's native grid is a curvilinear sigma-coordinate
  grid; `t3` = "tier 3" in SOMISANA's processing chain, meaning it has been
  interpolated onto fixed z-levels and a regular horizontal grid, which makes it
  much easier to work with).
- `croco_avg_t3_day.nc` — the same fields **averaged to daily means** (created
  here with `cdo dayavg`, see the file's `history` attribute). This is the file
  you will want for most exploratory work, since it's ~20x smaller and daily
  averages are usually what's needed for a forecast overview.

To keep the download small, this notebook only fetches the **daily-mean**
files (`croco_avg_t3_day.nc`, ~500 MB in total across both regions and all
four configurations) — the raw hourly files are many times larger and aren't
needed for anything we do here.

### Learning objectives

By the end of this notebook you will be able to:

1. Open and interpret the structure of a SOMISANA forecast file (dimensions,
   coordinates, variables, metadata).
2. Make maps of sea-surface and bottom fields with `cartopy`.
3. Compare the four nesting/forcing configurations against each other, to
   separate the effect of the **ocean boundary conditions** from the effect of
   the **atmospheric forcing**.
4. Look at the vertical structure of the water column and how it evolves over
   the forecast period.
5. Compute a simple derived product (surface current speed) from the model
   output.


## 1. Setup

In [ ]:
# Run once per environment (e.g. once per Colab session, or once in your local conda/venv)
%pip install -q numpy xarray matplotlib cartopy netCDF4 pathlib


In [ ]:

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

%matplotlib inline
plt.rcParams["figure.dpi"] = 100


In [ ]:
# The forecast files live on the SOMISANA sigma2 server, mirrored under the
# same folder layout we use locally: <region>/<config>/croco_avg_t3_day.nc
BASE_URL = "https://ns9481k.web.sigma2.no/SOMISANA_forecasts"

# We'll download into a "SOMISANA_forecasts" folder created next to this notebook.
BASE_DIR = Path("SOMISANA_forecasts").resolve()

REGIONS = ["SA-West", "SA-East"]
CONFIGS = ["HYCOM-GFS", "MERCATOR-GFS", "HYCOM-SAWS", "MERCATOR-SAWS"]

def filepath(region, config):
    '''Return the local path to a SOMISANA daily-mean forecast file.'''
    return BASE_DIR / region / config / "croco_avg_t3_day.nc"

def remote_url(region, config):
    '''Return the URL of a SOMISANA daily-mean forecast file on the sigma2 server.'''
    return f"{BASE_URL}/{region}/{config}/croco_avg_t3_day.nc"


### Downloading the data

The daily-mean forecast files (`croco_avg_t3_day.nc`) for both regions and all
four configurations are hosted at
https://ns9481k.web.sigma2.no/SOMISANA_forecasts/, using the same
`<region>/<config>/` layout as `filepath()` above. The cell below uses
`wget` to fetch each one into `BASE_DIR`, skipping any file that's already
been downloaded (so re-running the cell is cheap and safe to interrupt/resume
across sessions). This is **~500 MB in total**. The download takes soem time.

**Note**: this requires `wget` to be installed and on your `PATH` (already
the case on Colab/most Linux systems; on macOS install it with
`brew install wget`).


In [ ]:
import subprocess

def download_file(region, config):
    '''Download one SOMISANA daily-mean forecast file with wget, unless it's already on disk.'''
    dest = filepath(region, config)
    if dest.exists():
        print(f"{dest.relative_to(BASE_DIR)}  already downloaded, skipping")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = remote_url(region, config)
    print(f"Downloading {url}\n  -> {dest}")
    subprocess.run(
        ["wget", "-q", "--show-progress", "-O", str(dest), url],
        check=True,
    )

# Fetch the daily-mean file for every region/configuration combination.
for region in REGIONS:
    for config in CONFIGS:
        download_file(region, config)


In [ ]:
# quick sanity check that everything downloaded correctly
for region in REGIONS:
    for config in CONFIGS:
        p = filepath(region, config)
        print(f"{p}  {'OK' if p.exists() else 'MISSING'}")


## 2. The forecast file

Let's open a single file and look at how it's organised. We'll use the
**SA-West / HYCOM-GFS** daily-averaged file.


In [ ]:

ds = xr.open_dataset(filepath("SA-West", "HYCOM-GFS"))
ds


A few things to note:

- **Dimensions**: `time`, `longitude`, `latitude`, `depth` (plus a `bnds`
  dimension used for time bounds). Because this is a *tier 3* product, the
  horizontal grid is a plain regular lon/lat grid rather than CROCO's native
  curvilinear grid.
- **`depth`** is a small set of fixed z-levels (surface + a few standard
  depths down to 1000 m), created by vertically interpolating the model's
  native sigma layers.
- **Variables ending in `_bottom`** (`temp_bottom`, `salt_bottom`, `u_bottom`,
  `v_bottom`) give the value in the model's bottom-most sigma layer — useful
  for e.g. benthic ecosystems or sediment tranport, where the
  fixed z-levels above might not reach the seabed in shallow water.
- **`u`/`v`** are the *baroclinic* current components (see their
  `standard_name`), not the total, which is the baroclinic + the barotropic current.
- Land points are masked as `NaN`.

Let's look at the metadata that documents exactly how this file was produced.


In [ ]:

for key, value in ds.attrs.items():
    print(f"{key:22s}: {value}")


The `source` attribute encodes the model configuration
(`C06_I99_HYCOM_GFS_TPXO10` — CROCO nested in HYCOM, forced with GFS
atmospherics and TPXO10 tides) and the operational cycle directory
(`.../ops/main/<cycle>/sa_west_02/...`), so you can always trace a field back
to exactly which run produced it. `time_coverage_start/end` and
`date_created` tell you the forecast window and when it was generated.

In [ ]:

print("Forecast variable list:")
for v in ds.data_vars:
    da = ds[v]
    units = da.attrs.get("units", "")
    long_name = da.attrs.get("long_name", "")
    print(f"  {v:12s} dims={da.dims!s:45s} units={units:8s} {long_name}")


## 3. From hourly output to daily averages

The raw model output (`croco_avg_t3.nc`) is hourly. Daily averaging removes high-frequency variability, most obviously the semi-diurnal and diurnal tidal signal,
leaving the slower, sub-tidal evolution of the forecast.

We won't download the hourly files in this notebook — each one is ~24x larger than its daily-mean equivalent (multiple GB per configuration), which adds up fast across two regions and four configurations.

## 4. Mapping a forecast field

We'll plot sea-surface temperature (SST) for the first forecast day of the 
SA-West / HYCOM-GFS run, using `cartopy` for a correctly-projected coastline.


In [ ]:

def plot_field(ax, da, cmap="RdYlBu_r", vmin=None, vmax=None, add_colorbar=True, **kwargs):
    '''Convenience wrapper to plot a 2D (lat, lon) DataArray with cartopy coastlines.'''
    im = ax.pcolormesh(
        da.longitude, da.latitude, da,
        transform=ccrs.PlateCarree(), cmap=cmap, vmin=vmin, vmax=vmax, shading="auto", **kwargs
    )
    ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=2)
    ax.coastlines(resolution="10m", linewidth=0.7, zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.5", alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    if add_colorbar:
        plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    return im


In [ ]:

ds_west = xr.open_dataset(filepath("SA-West", "HYCOM-GFS"))
sst = ds_west.temp.isel(time=5, depth=0)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
im = plot_field(ax, sst, vmin=12, vmax=22)
ax.set_title(
    f"SST — SA-West / HYCOM-GFS\n{pd.Timestamp(sst.time.values).date()} (daily mean)"
)
plt.colorbar(im, ax=ax).remove()  # avoid duplicate colorbar from plot_field default
im2 = plot_field(ax, sst, vmin=12, vmax=22, add_colorbar=False)
fig.tight_layout()


You can clearly see the cold waters of the Benguela upwelling system hugging the coast.

## 5. Comparing the four forecast configurations

For the *same* region and the *same* day, how much does the forecast change depending on
which global ocean model supplies the boundary conditions (HYCOM vs.
Mercator) and which atmospheric product forces the surface (GFS vs. SAWS)?

We'll load SST for all four SA-West configurations on the first forecast day
and plot them side by side on a common colour scale.


In [ ]:

sst_by_config = {}
for config in CONFIGS:
    ds_c = xr.open_dataset(filepath("SA-West", config))
    sst_by_config[config] = ds_c.temp.isel(time=5, depth=0).load()

vmin, vmax = 12, 22
fig, axes = plt.subplots(
    2, 2, figsize=(11, 10), subplot_kw={"projection": ccrs.PlateCarree()}
)
for ax, config in zip(axes.flat, CONFIGS):
    im = plot_field(ax, sst_by_config[config], vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_title(config)

fig.suptitle(
    f"SST, SA-West, {pd.Timestamp(sst_by_config[CONFIGS[0]].time.values).date()} "
    "— all four nesting/forcing combinations",
    y=0.98,
)
cax = fig.add_axes([0.25, 0.04, 0.5, 0.02])
fig.colorbar(im, cax=cax, orientation="horizontal", label="Sea surface temperature (°C)")
fig.tight_layout(rect=[0, 0.06, 1, 0.96])


### The effect of the ocean nesting model vs. the atmospheric forcing

We can separate the two sources of forecast spread by taking differences:

- **HYCOM-GFS − MERCATOR-GFS**: same atmospheric forcing, different ocean
  boundary conditions → isolates the effect of the **nesting/boundary
  model**.
- **HYCOM-GFS − HYCOM-SAWS**: same ocean boundary conditions, different
  atmospheric forcing → isolates the effect of the **atmospheric forcing**.


In [ ]:

diff_ocean = sst_by_config["HYCOM-GFS"] - sst_by_config["MERCATOR-GFS"]
diff_atmos = sst_by_config["HYCOM-GFS"] - sst_by_config["HYCOM-SAWS"]

dmax = float(np.nanmax(np.abs([diff_ocean, diff_atmos])))

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), subplot_kw={"projection": ccrs.PlateCarree()})
im0 = plot_field(axes[0], diff_ocean, cmap="RdBu_r", vmin=-dmax, vmax=dmax, add_colorbar=False)
axes[0].set_title("Ocean nesting effect\nHYCOM-GFS minus MERCATOR-GFS")
im1 = plot_field(axes[1], diff_atmos, cmap="RdBu_r", vmin=-dmax, vmax=dmax, add_colorbar=False)
axes[1].set_title("Atmospheric forcing effect\nHYCOM-GFS minus HYCOM-SAWS")

cax = fig.add_axes([0.25, 0.02, 0.5, 0.03])
fig.colorbar(im1, cax=cax, orientation="horizontal", label="temperature difference (°C)")
fig.tight_layout(rect=[0, 0.07, 1, 1])

print(f"RMS difference from ocean nesting choice:      {float(np.sqrt(np.nanmean(diff_ocean**2))):.3f} degC")
print(f"RMS difference from atmospheric forcing choice: {float(np.sqrt(np.nanmean(diff_atmos**2))):.3f} degC")


On this forecast day, compare the RMS values printed above and the
spatial pattern of each difference map — is the SST forecast here more
sensitive to the choice of boundary/nesting model, or to the choice of
atmospheric forcing? 

**Exercises**

Try re-running from section 5 with a *later* forecast day, by changing `time=5` to a larger index above — does the answer change as the forecast lead time grows? Note that the actual forecast start only on day 5 (time=5).

Try re-running from section 5 with a *deeper* layer, by changing `depth=0` to a larger index above — does the answer change with depth? 

Add a new cells that make the same plots and RMS calculations for salinity



## 6. Vertical structure

The tier-3 files give us a handful of fixed depth levels, which is enough to
look at how temperature and salinity vary with depth, and how consistent
that structure is across the four configurations.

Pick a latitude and longitude below — the code will find the closest grid
point on the SA-West grid and plot the temperature and salinity profiles
there for all four configurations. Try an offshore point first (e.g. around
-33.0, 16.5, well out on the shelf); points closer to the coast can fall on
land and come back as NaN.


In [ ]:

# Choose a profile location — the closest grid point on the SA-West grid
# will be used for the profiles below.
lat_choice = float(input("Latitude (°N, e.g. -33.0): "))
lon_choice = float(input("Longitude (°E, e.g. 16.5): "))
time_choice = int(input("Forecast day (e.g. 5): "))

lat_idx = int(np.abs(ds_west.latitude - lat_choice).argmin())
lon_idx = int(np.abs(ds_west.longitude - lon_choice).argmin())
lat_pt = float(ds_west.latitude[lat_idx])
lon_pt = float(ds_west.longitude[lon_idx])
print(
    f"Closest grid point to ({lat_choice:.2f}, {lon_choice:.2f}): "
    f"{lat_pt:.2f} N, {lon_pt:.2f} E"
)

fig, axes = plt.subplots(1, 2, figsize=(9, 5), sharey=True)
any_data = False
for config in CONFIGS:
    ds_c = xr.open_dataset(filepath("SA-West", config))
    temp_profile = ds_c.temp.isel(time=time_choice, latitude=lat_idx, longitude=lon_idx).load()
    salt_profile = ds_c.salt.isel(time=time_choice, latitude=lat_idx, longitude=lon_idx).load()
    if np.isnan(temp_profile).all():
        continue
    any_data = True
    axes[0].plot(temp_profile, temp_profile.depth, "o-", label=config)
    axes[1].plot(salt_profile, salt_profile.depth, "o-", label=config)

if not any_data:
    print("No data at this grid point (likely on land) — try a point further offshore.")

axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Depth (m)")
axes[1].set_xlabel("Salinity")
axes[0].set_title("Temperature profile")
axes[1].set_title("Salinity profile")
axes[1].legend(loc="lower right", fontsize=8)
fig.suptitle(f"Vertical structure at {lat_pt:.2f}°S, {lon_pt:.2f}°E — forecast day 1")
fig.tight_layout()


Note that the fixed z-levels used here (0, 5, 10, 50, 100, 500, 1000 m)
are coarse compared to CROCO's native sigma layers — fine for a broad look at
stratification, but for a detailed look at the vertical structure you should use the full vertical resolution.


## 7. Forecast evolution over the lead time

Try to track SST at our profile location over the whole forecast window for all four
configurations.

Note: the GFS-forced runs provide a 5-day forecast while the SAWS-forced runs are 3 days. We'll plot each series over its own full length (hindcast and forecast), so this is visible directly in the plot.


In [ ]:

fig, ax = plt.subplots(figsize=(9, 4.5))
for config in CONFIGS:
    ds_c = xr.open_dataset(filepath("SA-West", config))
    sst_ts = ds_c.temp.isel(depth=0, latitude=lat_idx, longitude=lon_idx).load()
    ax.plot(sst_ts.time, sst_ts, "o-", label=f"{config} ({sst_ts.sizes['time']} days)")

ax.set_ylabel("Sea surface temperature (°C)")
ax.set_title(f"SST forecast evolution at {lat_pt:.2f}°S, {lon_pt:.2f}°E")
ax.legend(fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()


The spread between the four lines at a given date is a rough estimate of forecast uncertainty at that lead time — with only four members and no formal perturbation strategy this is *not* a real ensemble forecast, but the same principle (running multiple configurations/forcings and looking at their spread) underlies genuine
ensemble ocean forecasting systems.


## 8. Surface currents

Compute current speed from the baroclinic velocity components and visualise both
speed and direction.


In [ ]:

u = ds_west.u.isel(time=0, depth=0)
v = ds_west.v.isel(time=0, depth=0)
speed = np.sqrt(u**2 + v**2)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
im = plot_field(ax, speed, cmap="viridis", vmin=0, add_colorbar=False)
plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02, label="Current speed (m/s)")

# subsample the vectors so the quiver plot is readable
step = 8
ax.quiver(
    u.longitude[::step], u.latitude[::step],
    u.values[::step, ::step], v.values[::step, ::step],
    transform=ccrs.PlateCarree(), scale=8, width=0.003, color="w",
)
ax.set_title(
    f"Baroclinic surface currents — SA-West / HYCOM-GFS\n{pd.Timestamp(u.time.values).date()}"
)
fig.tight_layout()


## 9. Bottom temperature

The `*_bottom` variables sample the model's bottom-most sigma layer rather
than a fixed z-level, so they follow the seabed even in shallow coastal
water where the 0/5/10 m z-levels above might already be below the bottom.
This is often the field of interest for e.g. benthic habitat studies.


In [ ]:

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
im = plot_field(ax, ds_west.temp_bottom.isel(time=0), vmin=8, vmax=20, add_colorbar=False)
plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02, label="Bottom temperature (°C)")
ax.set_title(
    f"Bottom temperature — SA-West / HYCOM-GFS\n{pd.Timestamp(ds_west.time.isel(time=0).values).date()}"
)
fig.tight_layout()


Compare this to the surface temperature maps made in Section 4 — the cold band near the coast versus the bottom temperature here.


## Summary and suggested exercises

You've now explored some data of SOMISANA's South West operational CROCO forecasts, and used the four HYCOM/MERCATOR × GFS/SAWS configurations to get a feel for how sensitive a short-range ocean forecast is to its boundary and atmospheric forcing.

Some ideas to extend this notebook:

1. **Sout East forecast** Repat the analysis above for the forecast regions in the South East.
2. **Time–depth sections**: pick a point and plot a Hovmöller diagram of
   temperature or salinity vs. depth vs. forecast day, to see how it evolves.
3. **A transect**: extract a cross-shelf section (fixed latitude, varying
   longitude and depth) to look at the upwelling front in cross-section.
4. **Compare `u`/`v` totals**: SOMISANA's other tiers of output also include
   the barotropic component — see if you can find and add it to get the total
   current. Tips: See below for how to use the Tredds server to download a subset of data.
5. **Fetch a fresh forecast**: the files here are a single snapshot from 3. September 2026
   https://catalog.somisana.ac.za/catalog/. Try downloading the forecast from 6. September
   and compare the two forecasts on the same day have in common? (This is called forecast    *consistency*, a standard operational-forecasting diagnostic.) See the appendix below     for how to pull a subset of a fresh cycle straight over OPeNDAP instead of
   downloading the whole file.


## Appendix: Downloading a custom subset via OPeNDAP

So far we've downloaded whole daily forecast files with `wget`. For a quick look at
a small region, a single depth level, or just the first day or two of a
forecast, it's usually faster to use **OPeNDAP** instead: it lets `xarray`
open a NetCDF file that lives on a remote server and read only the piece you
actually ask for, over the network, without downloading the whole file first.

**Finding a dataset's OPeNDAP address, using the browser:**

1. Go to the SOMISANA catalog: https://catalog.somisana.ac.za/catalog/ and
   open one of the four products (e.g. *SA-West Forecast V1.0*).
2. On the product page, follow the **Thredds server** link.
3. Browse into a forecast-cycle folder, named `YYYYMMDD_00` (e.g. `20260906_00`
   for the run started at 00Z on 6 September) — pick the most recent one, or
   an older one to look back at a past forecast.
4. Open one of the four model-configuration folders (`HYCOM-GFS`,
   `HYCOM-SAWS`, `MERCATOR-GFS`, `MERCATOR-SAWS`).
5. Click on `croco_avg_t3.nc` — the same 7-fixed-depth-level product we've
   used throughout this notebook. Note that the copy on the server is the
   full hourly forecast (not the daily-averaged version we downloaded
   earlier), so it has more timesteps.
6. On the dataset's access page, click **OpenDAP**. The link shown there ends
   in `.html`; the actual data URL for `xarray` is the same address with
   `.html` removed.


In [ ]:

# Region folder names on the OPeNDAP server (note: "sa-southeast", not "SA-East")
OPENDAP_REGIONS = {"SA-West": "sa-west", "SA-East": "sa-southeast"}
OPENDAP_BASE = "https://somisana.ocean.gov.za/thredds/dodsC/mnt/ocims-somisana/public-facing"

def opendap_url(region, config, cycle, filename="croco_avg_t3.nc"):
    '''Build the OPeNDAP URL for a SOMISANA forecast file.

    `cycle` is the forecast-cycle folder name found by browsing the Thredds
    catalog above, e.g. "20260906_00".
    '''
    server_region = OPENDAP_REGIONS[region]
    return f"{OPENDAP_BASE}/{server_region}/v1.0/forecasts/{cycle}/{config}/{filename}"


# Update `cycle` to a folder name you found by browsing the Thredds catalog
cycle = "20260906_00"
url = opendap_url("SA-West", "HYCOM-GFS", cycle)
print(url)

ds_remote = xr.open_dataset(url)
ds_remote


Opening the dataset this way doesn't download anything yet — `xarray` only
reads the metadata (dimensions, coordinates, variable names) over the
network. The data itself is only transferred once you select a subset and
explicitly load it, e.g. with `.load()` or `.to_netcdf()`.

Let's pull out a small bounding box around Cape Town, the surface layer only,
and the first day of the forecast, and save that subset to a local file:


In [ ]:

subset = ds_remote[["temp", "salt"]].sel(
    latitude=slice(-35, -33),
    longitude=slice(17.5, 19.5),
    depth=0,
).isel(time=slice(0, 24))  # first 24 hourly steps = 1 day

subset = subset.load()  # this is the point where data actually gets transferred
subset.to_netcdf(BASE_DIR / "sa_west_cape_town_subset.nc")
subset


Because latitude and longitude both increase with index in these files, a
plain `slice(min, max)` works directly with `.sel()`. If you ever work with a
dataset where a coordinate decreases with index, you'd need to swap the
slice order, e.g. `slice(max, min)`.

Try adapting the code above to your own region of interest, a different
depth level, or a longer time window — this is how you can get a
manageable, task-specific subset of an operational forecast without
downloading the entire file.
